# 🧠 MRI Brain Tumor MLOps Pipeline
> ResNet50 Classifier + ResUNet Segmentation | Kaggle Data | MLflow Tracking

**Runtime → Change runtime type → T4 GPU** before running!

## ✅ Step 1 — Check GPU

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))
!nvidia-smi

## ✅ Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/mri-brain-tumor-mlops'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

## ✅ Step 3 — Upload & Unzip Project Files

In [ ]:
# Upload your mri-brain-tumor-mlops.zip here
from google.colab import files
uploaded = files.upload()  # select mri-brain-tumor-mlops.zip

import zipfile
for fname in uploaded:
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/drive/MyDrive/')
    print(f'Extracted: {fname}')

os.chdir(PROJECT_DIR)
!ls -la

## ✅ Step 4 — Install Dependencies

In [ ]:
%%capture
!pip install mlflow evidently gradio fastapi uvicorn pyyaml opencv-python-headless kaggle
print('✅ All packages installed')

## ✅ Step 5 — Download Kaggle Dataset (LGG MRI)

In [ ]:
# ----------------------------------------------------------
# Option A: Upload kaggle.json manually
# ----------------------------------------------------------
from google.colab import files
print('Upload your kaggle.json API token:')
uploaded = files.upload()  # Upload kaggle.json from https://www.kaggle.com/settings

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('✅ Kaggle credentials configured')

In [ ]:
# Download LGG MRI Segmentation dataset
!kaggle datasets download -d mateuszbuda/lgg-mri-segmentation -p data/raw/ --unzip
print('✅ Dataset downloaded')
!ls data/raw/

## ✅ Step 6 — Verify Data Structure

In [ ]:
import pandas as pd
from pathlib import Path
import cv2
import sys
sys.path.insert(0, PROJECT_DIR)

records = []
for img_path in sorted(Path('data/raw').rglob('*.tif')):
    if '_mask' not in img_path.name:
        mask_path = img_path.parent / (img_path.stem + '_mask' + img_path.suffix)
        if mask_path.exists():
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            has_mask = int(mask.max() > 0)
            records.append({'image_path': str(img_path),
                             'mask_path': str(mask_path),
                             'has_mask': has_mask})

df = pd.DataFrame(records)
print(f'Total samples    : {len(df)}')
print(f'Tumor positive   : {df["has_mask"].sum()}')
print(f'Tumor negative   : {(df["has_mask"]==0).sum()}')
print(f'Positive ratio   : {df["has_mask"].mean():.2%}')
df.to_csv('data/processed/full_dataset.csv', index=False)
df.head()

## ✅ Step 7 — Start MLflow (Background)

In [ ]:
import subprocess, time
os.makedirs('logs', exist_ok=True)
mlflow_proc = subprocess.Popen(
    ['mlflow', 'ui', '--host', '0.0.0.0', '--port', '5000',
     '--backend-store-uri', f'sqlite:///{PROJECT_DIR}/mlflow.db',
     '--default-artifact-root', f'{PROJECT_DIR}/mlflow_artifacts'],
    stdout=open('logs/mlflow.log', 'w'),
    stderr=subprocess.STDOUT
)
time.sleep(3)
print('✅ MLflow started')

# Expose via ngrok for browser access
!pip install -q pyngrok
from pyngrok import ngrok
ngrok.kill()
public_url = ngrok.connect(5000)
print(f'🔗 MLflow UI: {public_url}')

## ✅ Step 8 — Train Classifier (ResNet50)

In [ ]:
import mlflow
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import AveragePooling2D, Dense, Dropout, Flatten, Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)

# --- Config ---
IMG_SIZE = (256, 256)
BATCH_SIZE = 16
EPOCHS = 60
LR = 1e-4

# --- Split ---
df['has_mask'] = df['has_mask'].astype(str)
train_val, test_df = train_test_split(df, test_size=0.15, stratify=df['has_mask'], random_state=42)
train_df, val_df = train_test_split(train_val, test_size=0.176, stratify=train_val['has_mask'], random_state=42)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# --- Generators ---
train_aug = ImageDataGenerator(rescale=1./255, rotation_range=10, width_shift_range=0.1,
    height_shift_range=0.1, zoom_range=0.1, horizontal_flip=True, fill_mode='nearest')
val_aug = ImageDataGenerator(rescale=1./255)

train_gen = train_aug.flow_from_dataframe(train_df, x_col='image_path', y_col='has_mask',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True)
val_gen = val_aug.flow_from_dataframe(val_df, x_col='image_path', y_col='has_mask',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
test_gen = val_aug.flow_from_dataframe(test_df, x_col='image_path', y_col='has_mask',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

# --- Model ---
base = ResNet50(weights='imagenet', include_top=False, input_tensor=Input(shape=(256,256,3)))
for layer in base.layers:
    layer.trainable = False
x = AveragePooling2D((4,4))(base.output)
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
out = Dense(2, activation='softmax')(x)
clf_model = Model(inputs=base.input, outputs=out)

clf_model.compile(optimizer=tf.keras.optimizers.Adam(LR),
                  loss='categorical_crossentropy', metrics=['accuracy'])

# --- MLflow Run ---
mlflow.set_tracking_uri(f'sqlite:///{PROJECT_DIR}/mlflow.db')
mlflow.set_experiment('brain_tumor_classifier')

with mlflow.start_run(run_name='resnet50_classifier'):
    mlflow.log_params({'backbone':'ResNet50','lr':LR,'batch_size':BATCH_SIZE,'epochs':EPOCHS})

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
        ModelCheckpoint('models/classifier-resnet-weights.keras', monitor='val_loss',
                         save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5, verbose=1)
    ]

    history = clf_model.fit(
        train_gen,
        steps_per_epoch=train_gen.n // BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=val_gen,
        validation_steps=val_gen.n // BATCH_SIZE,
        callbacks=callbacks
    )

    best_val_acc = max(history.history['val_accuracy'])
    best_val_loss = min(history.history['val_loss'])
    mlflow.log_metrics({'best_val_accuracy': best_val_acc, 'best_val_loss': best_val_loss})
    mlflow.keras.log_model(clf_model, 'classifier_model',
                            registered_model_name='brain_tumor_classifier')
    print(f'\n✅ Classifier training done. Best val_accuracy: {best_val_acc:.4f}')

## ✅ Step 9 — Train Segmentation Model (ResUNet)

In [ ]:
import tensorflow.keras.backend as K
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Add
from tensorflow.keras.layers import Conv2DTranspose, concatenate, MaxPooling2D

# --- Focal Tversky Loss ---
def tversky_score(y_true, y_pred, alpha=0.7, beta=0.3, smooth=1.):
    y_true_f = K.flatten(K.cast(y_true, tf.float32))
    y_pred_f = K.flatten(y_pred)
    tp = K.sum(y_true_f * y_pred_f)
    fp = K.sum((1 - y_true_f) * y_pred_f)
    fn = K.sum(y_true_f * (1 - y_pred_f))
    return (tp + smooth) / (tp + alpha*fn + beta*fp + smooth)

def focal_tversky_loss(y_true, y_pred, gamma=0.75):
    return K.pow((1 - tversky_score(y_true, y_pred)), gamma)

# --- ResUNet Architecture ---
def residual_block(x, filters):
    shortcut = Conv2D(filters, 1, padding='same')(x)
    shortcut = BatchNormalization()(shortcut)
    x = Conv2D(filters, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([x, shortcut])
    return Activation('relu')(x)

def build_resunet():
    inp = Input((256, 256, 3))
    e1 = residual_block(inp, 32);   p1 = MaxPooling2D()(e1)
    e2 = residual_block(p1, 64);    p2 = MaxPooling2D()(e2)
    e3 = residual_block(p2, 128);   p3 = MaxPooling2D()(e3)
    e4 = residual_block(p3, 256);   p4 = MaxPooling2D()(e4)
    b  = residual_block(p4, 512)
    d4 = Conv2DTranspose(256,(2,2),strides=2,padding='same')(b)
    d4 = residual_block(concatenate([d4,e4]), 256)
    d3 = Conv2DTranspose(128,(2,2),strides=2,padding='same')(d4)
    d3 = residual_block(concatenate([d3,e3]), 128)
    d2 = Conv2DTranspose(64,(2,2),strides=2,padding='same')(d3)
    d2 = residual_block(concatenate([d2,e2]), 64)
    d1 = Conv2DTranspose(32,(2,2),strides=2,padding='same')(d2)
    d1 = residual_block(concatenate([d1,e1]), 32)
    out = Conv2D(1, 1, activation='sigmoid')(d1)
    return Model(inp, out, name='ResUNet')

seg_model = build_resunet()
seg_model.compile(optimizer=tf.keras.optimizers.Adam(lr=0.05, epsilon=0.1),
                  loss=focal_tversky_loss, metrics=[tversky_score])
print(f'ResUNet params: {seg_model.count_params():,}')

# --- Segmentation Generator (only positive samples) ---
positive_df = df[df['has_mask'] == '1'].copy().reset_index(drop=True)
seg_train, seg_temp = train_test_split(positive_df, test_size=0.3, random_state=42)
seg_val, seg_test   = train_test_split(seg_temp,     test_size=0.5, random_state=42)
print(f'Seg train: {len(seg_train)} | val: {len(seg_val)} | test: {len(seg_test)}')

class SegGen(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=16, augment=False):
        self.df = df.reset_index(drop=True)
        self.bs = batch_size
        self.augment = augment
    def __len__(self): return len(self.df) // self.bs
    def __getitem__(self, idx):
        batch = self.df.iloc[idx*self.bs:(idx+1)*self.bs]
        imgs, masks = [], []
        for _, row in batch.iterrows():
            img = cv2.imread(row['image_path'])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (256,256)).astype(np.float32) / 255.
            mask = cv2.imread(row['mask_path'], cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (256,256)).astype(np.float32) / 255.
            imgs.append(img)
            masks.append(np.expand_dims(mask, -1))
        return np.array(imgs), np.array(masks)

# --- Train ResUNet with MLflow ---
mlflow.set_experiment('brain_tumor_segmentation')

with mlflow.start_run(run_name='resunet_segmentation'):
    mlflow.log_params({'model':'ResUNet','lr':0.05,'epochs':60,
                        'loss':'focal_tversky','tversky_alpha':0.7,'tversky_gamma':0.75})

    seg_callbacks = [
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
        ModelCheckpoint('models/ResUNet-weights.keras', monitor='val_loss',
                         save_best_only=True, verbose=1)
    ]

    seg_history = seg_model.fit(
        SegGen(seg_train, augment=True),
        epochs=60,
        validation_data=SegGen(seg_val),
        callbacks=seg_callbacks
    )

    best_tv = max(seg_history.history['val_tversky_score'])
    mlflow.log_metric('best_val_tversky', best_tv)
    mlflow.keras.log_model(seg_model, 'resunet_model',
                            registered_model_name='brain_tumor_segmentor')
    print(f'\n✅ Segmentation training done. Best val_tversky: {best_tv:.4f}')

## ✅ Step 10 — Evaluate Both Models

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# --- Classifier Evaluation ---
test_preds_prob = clf_model.predict(test_gen, steps=test_gen.n // BATCH_SIZE)
test_preds = np.argmax(test_preds_prob, axis=1)
y_true = test_gen.classes[:len(test_preds)]

print('=== Classifier Report ===')
print(classification_report(y_true, test_preds, target_names=['No Tumor','Tumor']))
auc = roc_auc_score(y_true, test_preds_prob[:,1])
print(f'AUC: {auc:.4f}')

# --- Segmentation Evaluation ---
def dice_score(y_true, y_pred):
    y_true_f, y_pred_f = y_true.flatten(), (y_pred.flatten() > 0.5).astype(float)
    return (2.*np.sum(y_true_f*y_pred_f)+1.) / (np.sum(y_true_f)+np.sum(y_pred_f)+1.)

def iou_score(y_true, y_pred):
    y_true_f, y_pred_f = y_true.flatten(), (y_pred.flatten() > 0.5).astype(float)
    inter = np.sum(y_true_f*y_pred_f)
    return (inter+1.) / (np.sum(y_true_f)+np.sum(y_pred_f)-inter+1.)

dice_scores, iou_scores = [], []
test_seg_gen = SegGen(seg_test)
for imgs, masks in test_seg_gen:
    preds = seg_model.predict(imgs, verbose=0)
    for gt, pr in zip(masks, preds):
        dice_scores.append(dice_score(gt, pr))
        iou_scores.append(iou_score(gt, pr))

print(f'\n=== Segmentation Evaluation ===')
print(f'Avg Dice Score : {np.mean(dice_scores):.4f}')
print(f'Avg IoU Score  : {np.mean(iou_scores):.4f}')

mlflow.log_metrics({'test_auc': auc, 'test_avg_dice': np.mean(dice_scores),
                     'test_avg_iou': np.mean(iou_scores)})

## ✅ Step 11 — Visualize Predictions

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
sample_gen = SegGen(seg_test, batch_size=4)
sample_imgs, sample_masks = sample_gen[0]
sample_preds = seg_model.predict(sample_imgs, verbose=0)

for i in range(4):
    axes[i][0].imshow(sample_imgs[i])
    axes[i][0].set_title('Input MRI')
    axes[i][0].axis('off')
    axes[i][1].imshow(sample_masks[i,:,:,0], cmap='gray')
    axes[i][1].set_title('Ground Truth Mask')
    axes[i][1].axis('off')
    axes[i][2].imshow(sample_preds[i,:,:,0] > 0.5, cmap='Reds')
    axes[i][2].set_title(f'Predicted Mask (Dice={dice_score(sample_masks[i], sample_preds[i]):.3f})')
    axes[i][2].axis('off')

plt.suptitle('ResUNet Segmentation Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/segmentation_predictions.png', dpi=150)
plt.show()
print('✅ Saved to reports/segmentation_predictions.png')

## ✅ Step 12 — Save Everything to Google Drive

In [ ]:
import shutil

# Models are already saved to PROJECT_DIR/models/ via ModelCheckpoint
# Save the dataset CSV
df.to_csv(f'{PROJECT_DIR}/data/processed/full_dataset.csv', index=False)

print('✅ All files saved to Google Drive:')
for f in ['models/classifier-resnet-weights.keras',
           'models/ResUNet-weights.keras',
           'reports/segmentation_predictions.png',
           'data/processed/full_dataset.csv',
           'mlflow.db']:
    full = f'{PROJECT_DIR}/{f}'
    exists = os.path.exists(full)
    size = os.path.getsize(full)/1e6 if exists else 0
    print(f'  {"✅" if exists else "❌"} {f} ({size:.1f} MB)')